# 模块 01：滤波与图像增强

图像滤波是CV中最基础也最常用的操作。从手机"美颜"到医学图像降噪，背后都是滤波算法。

本模块涵盖：卷积原理、平滑滤波器、直方图均衡化、锐化技术

> 预计学习时间：60-90 分钟

## 学习目标

- 理解卷积操作的核心原理
- 掌握高斯模糊、中值滤波、双边滤波的区别和适用场景
- 学会直方图均衡化和 CLAHE
- 能实现 USM 锐化
- 了解"老照片修复"的完整流程

## 1. 卷积的物理意义

> 卷积核在图像上滑动，每个位置的输出 = 核与对应像素的加权求和

不同核值产生不同效果：
- 核值全为正且和为1 → 模糊（加权平均）
- 核中心为正、周围为负 → 锐化（增强差异）
- 核有正有负且和为0 → 边缘检测

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, filters, util, exposure, color, morphology
from scipy import ndimage
import cv2
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

img = data.camera()
img_color = data.astronaut()
print("Sample images loaded")
print(f"camera: {img.shape}, dtype={img.dtype}")
print(f"astronaut: {img_color.shape}, dtype={img_color.dtype}")

## 2. 添加噪声——展示滤波器效果

先给图片添加噪声，才能看到滤波器的作用：

In [ ]:
def add_salt_pepper(image, amount=0.05):
    noisy = image.copy()
    h, w = image.shape[:2]
    num_sp = int(amount * h * w)
    coords = np.random.randint(0, max(h,w), (num_sp, 2))
    for y, x in coords:
        if y < h and x < w:
            noisy[y, x] = 0 if np.random.random() < 0.5 else 255
    return noisy

def add_gaussian_noise(image, sigma=25):
    noise = np.random.normal(0, sigma, image.shape)
    return np.clip(image.astype(float) + noise, 0, 255).astype(np.uint8)

img_sp = add_salt_pepper(img, amount=0.05)
img_gauss = add_gaussian_noise(img, sigma=25)

fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
axes[1].imshow(img_sp, cmap='gray'); axes[1].set_title('Salt & Pepper (5%)',fontsize=13); axes[1].axis('off')
axes[2].imshow(img_gauss, cmap='gray'); axes[2].set_title('Gaussian (sigma=25)',fontsize=13); axes[2].axis('off')
plt.tight_layout()
plt.show()

## 3. 高斯模糊

$$G(x,y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}}$$

关键参数 sigma：越大越模糊。

In [ ]:
sigmas = [0.5, 1, 2, 4]
fig, axes = plt.subplots(1, len(sigmas)+1, figsize=(18,4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
for i, s in enumerate(sigmas):
    blurred = filters.gaussian(img, sigma=s)
    axes[i+1].imshow(blurred, cmap='gray')
    axes[i+1].set_title(f'sigma={s}',fontsize=13); axes[i+1].axis('off')
plt.suptitle('Gaussian Blur - Larger sigma = More blur',fontsize=15,y=1.05)
plt.tight_layout()
plt.show()

## 4. 中值滤波——椒盐噪声克星

中值滤波用邻域的**中位数**替换中心像素。椒盐噪声的极端值(0或255)不影响中位数，因此可以完全清除而保留边缘。

In [ ]:
img_median = ndimage.median_filter(img_sp, size=3)
img_gauss_on_sp = filters.gaussian(img_sp, sigma=1)

fig, axes = plt.subplots(1, 4, figsize=(18,5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
axes[1].imshow(img_sp, cmap='gray'); axes[1].set_title('Salt & Pepper',fontsize=13); axes[1].axis('off')
axes[2].imshow(img_median, cmap='gray'); axes[2].set_title('Median 3x3 - Clean!',fontsize=13); axes[2].axis('off')
axes[3].imshow(img_gauss_on_sp, cmap='gray'); axes[3].set_title('Gaussian on S&P - Still noisy',fontsize=13); axes[3].axis('off')
plt.tight_layout()
plt.show()
print("Median filter: noise replaced by neighbor median")
print("Gaussian blur: noise just blurred, not removed")

## 5. 双边滤波——美颜核心算法

双边滤波 = 高斯模糊 + 边缘保护。它同时考虑：
1. **空间距离**（近的权重大）
2. **像素值差异**（颜色相近的权重大——关键创新）

结果：平坦区域平滑，边缘保持清晰。这是手机美颜"磨皮"的核心原理。

In [ ]:
img_c_noisy = add_gaussian_noise(img_color, sigma=20)

# Bilateral filter (OpenCV)
img_bilateral = cv2.bilateralFilter(img_c_noisy, d=9, sigmaColor=75, sigmaSpace=75)

# Gaussian for comparison
img_gauss_c = np.zeros_like(img_c_noisy)
for c in range(3):
    img_gauss_c[:,:,c] = filters.gaussian(img_c_noisy[:,:,c], sigma=2)

fig, axes = plt.subplots(1, 4, figsize=(18,5))
axes[0].imshow(img_color); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
axes[1].imshow(img_c_noisy); axes[1].set_title('Gaussian noise sigma=20',fontsize=13); axes[1].axis('off')
axes[2].imshow(img_gauss_c); axes[2].set_title('Gaussian blur - Edges blurry',fontsize=13); axes[2].axis('off')
axes[3].imshow(img_bilateral); axes[3].set_title('Bilateral - Edges preserved!',fontsize=13); axes[3].axis('off')
plt.tight_layout()
plt.show()
print("Bilateral filter = core of beauty mode algorithm")

## 6. 直方图均衡化 & CLAHE

直方图均衡化重新分布像素值增强对比度。CLAHE (自适应) 将图像分块独立均衡后拼接，比全局均衡更自然。

In [ ]:
img_dark = np.clip(data.camera().astype(float)*0.4, 0,255).astype(np.uint8)

img_eq = exposure.equalize_hist(img_dark)
img_eq = (img_eq * 255).astype(np.uint8)

img_clahe = exposure.equalize_adapthist(img_dark, clip_limit=0.03)
img_clahe = (img_clahe * 255).astype(np.uint8)

fig, axes = plt.subplots(2, 4, figsize=(18,8))
titles = ['Dark Original','Global Equalization','CLAHE','CLAHE (stronger)']
images = [img_dark, img_eq, img_clahe,
          (exposure.equalize_adapthist(img_dark,clip_limit=0.06)*255).astype(np.uint8)]
for ax,t,im in zip(axes[0],titles,images):
    ax.imshow(im,cmap='gray'); ax.set_title(t,fontsize=12); ax.axis('off')
for ax,im,t in zip(axes[1],images,titles):
    ax.hist(im.ravel(),bins=256,range=(0,255),color='gray',alpha=0.7)
    ax.set_title(f'Histogram: {t}',fontsize=10); ax.set_xlim(0,255)
plt.suptitle('Histogram Equalization Comparison',fontsize=15,y=1.02)
plt.tight_layout()
plt.show()
print("CLAHE = Contrast Limited Adaptive Histogram Equalization")
print("Divides image into tiles, equalizes each independently, then stitches")

## 7. USM 锐化

$$\text{Sharpened} = \text{Original} + \text{amount} \times (\text{Original} - \text{Blurred})$$

`原图 - 模糊图`提取高频细节(边缘)，amount控制增强程度。

In [ ]:
def usm_sharpen(image, radius=1.0, amount=1.0):
    blurred = filters.gaussian(image, sigma=radius, channel_axis=-1 if image.ndim==3 else None)
    sharpened = image + amount * (image - blurred)
    return np.clip(sharpened, 0, 255 if image.dtype==np.uint8 else 1)

img_f = img_color / 255.0
img_blurred = filters.gaussian(img_f, sigma=1.5, channel_axis=-1)
img_sharp = usm_sharpen(img_blurred, radius=1.5, amount=1.5)

fig, axes = plt.subplots(1, 3, figsize=(16,5))
axes[0].imshow(img_f); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
axes[1].imshow(np.clip(img_blurred,0,1)); axes[1].set_title('Blurred (sigma=1.5)',fontsize=13); axes[1].axis('off')
axes[2].imshow(np.clip(img_sharp,0,1)); axes[2].set_title('USM Sharpened\n(radius=1.5, amount=1.5)',fontsize=12); axes[2].axis('off')
plt.tight_layout()
plt.show()
print("USM cannot truly recover lost info, only enhances existing edge contrast")

## 🎮 交互式滤波探索

In [ ]:
def explore_filters(filter_type='gaussian', sigma=1.0, ksize=3, clip=0.03, amount=1.0):
    img = data.camera()
    if filter_type == 'gaussian':
        result = filters.gaussian(img, sigma=sigma); title = f'Gaussian sigma={sigma}'
    elif filter_type == 'median':
        result = ndimage.median_filter(img, size=max(3,ksize)); title = f'Median size={max(3,ksize)}'
    elif filter_type == 'clahe':
        result = (exposure.equalize_adapthist(img, clip_limit=clip)*255).astype(np.uint8); title = f'CLAHE clip={clip:.3f}'
    elif filter_type == 'sharpen':
        result = usm_sharpen(img/255.0, radius=sigma, amount=amount)
        result = (np.clip(result,0,1)*255).astype(np.uint8); title = f'USM sigma={sigma} amount={amount}'

    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5))
    ax1.imshow(img,cmap='gray'); ax1.set_title('Original',fontsize=13); ax1.axis('off')
    ax2.imshow(result,cmap='gray'); ax2.set_title(title,fontsize=13); ax2.axis('off')
    plt.show()

interact(explore_filters,
    filter_type=Dropdown(options=['gaussian','median','clahe','sharpen'],value='gaussian'),
    sigma=FloatSlider(min=0.1,max=5.0,step=0.1,value=1.0),
    ksize=IntSlider(min=3,max=11,step=2,value=3),
    clip=FloatSlider(min=0.01,max=0.1,step=0.005,value=0.03),
    amount=FloatSlider(min=0.1,max=3.0,step=0.1,value=1.0))
print("Select filter type and adjust parameters")

## 📝 应用案例：老照片修复

老照片三步修复：去噪 → 增强对比度 → 锐化

In [ ]:
# Simulate old photo
img_orig = data.camera()
old = add_gaussian_noise(img_orig, sigma=10)
old = (old * 0.7 + 30).astype(np.uint8)
old = filters.gaussian(old, sigma=0.8)

# Restoration pipeline
denoised = ndimage.median_filter(old, size=3)
enhanced = (exposure.equalize_adapthist(denoised, clip_limit=0.03)*255).astype(np.uint8)
restored = usm_sharpen(enhanced/255.0, radius=1.0, amount=0.8)
restored = (np.clip(restored,0,1)*255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(20,4))
steps = [(img_orig,'Original'),(old,'Old Photo\n(noise+low contrast)'),
         (denoised,'Step1: Denoise'),(enhanced,'Step2: CLAHE'),(restored,'Step3: Sharpen')]
for ax,(im,t) in zip(axes,steps):
    ax.imshow(im,cmap='gray'); ax.set_title(t,fontsize=11); ax.axis('off')
plt.suptitle('Old Photo Restoration Pipeline',fontsize=15,y=1.02)
plt.tight_layout()
plt.show()

## 🏋️ 动手练习

1. **噪声与滤波匹配**：对椒盐噪声和高斯噪声分别用高斯滤波和中值滤波处理，哪种组合最好？为什么？
2. **双边滤波参数实验**：调整 sigmaColor 和 sigmaSpace 参数观察各自影响
3. **CLAHE vs 全局均衡**：对同时有亮区和暗区的图片比较 CLAHE 和全局直方图均衡

In [ ]:
# TODO: Exercise 1 - Test 4 combinations
# Salt&Pepper + Gaussian, Salt&Pepper + Median, GaussianNoise + Gaussian, GaussianNoise + Median
# Visualize and analyze which combo works best

# TODO: Exercise 2 - Bilateral parameter experiment
# Try sigmaColor = [10, 50, 100, 150]

print("Complete the exercises above!")

## 📖 本节总结

| 滤波器 | 用途 | 特点 |
|--------|------|------|
| 高斯模糊 | 通用平滑 | 柔和模糊 |
| 中值滤波 | 椒盐噪声 | 保边 |
| 双边滤波 | 美颜/保边去噪 | 保边 |
| 直方图均衡 | 增强对比度 | 全局操作 |
| CLAHE | 自适应增强 | 分块自然 |
| USM锐化 | 增强细节 | 原图+高频 |

下一课：模块02 — 边缘检测与特征匹配